# Exemplo 06 - Problema da Mochila com DEAP

---

## O que e o DEAP?

**DEAP** (Distributed Evolutionary Algorithms in Python) e a biblioteca de AG mais usada na **academia**.

## PyGAD vs DEAP

| Aspecto | PyGAD (Ex 05) | DEAP (este) |
|:---|:---:|:---:|
| Filosofia | "Configure e rode" | "Monte peca por peca" |
| Artigos cientificos | Poucos | **Muito citado** |
| Flexibilidade | Media | **Altissima** |
| Multi-objetivo | Nao | **Sim** |
| Dificuldade | Baixa | Media |

## Conceitos novos do DEAP:

- **`creator`**: define os TIPOS (Fitness, Individuo)
- **`toolbox`**: registra os OPERADORES (selecao, crossover, mutacao)
- **`algorithms`**: possui AGs prontos (eaSimple, eaMuPlusLambda...)
- **`HallOfFame`**: guarda os N melhores individuos de todas as geracoes

In [1]:
# Instalar o DEAP (rodar apenas uma vez)
%pip install deap -q

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import random
from deap import base, creator, tools, algorithms

---
## Passo 1: Definir o problema (mesmo dos Exemplos 02 e 05)

In [13]:
itens = [
    {"nome": "Notebook",      "peso": 3.0, "valor": 2000},
    {"nome": "Livro de IA",   "peso": 1.5, "valor": 300},
    {"nome": "Garrafa agua",  "peso": 1.0, "valor": 50},
    {"nome": "Camera",        "peso": 2.0, "valor": 1500},
    {"nome": "Casaco",        "peso": 1.5, "valor": 200},
    {"nome": "Lanche",        "peso": 0.5, "valor": 80},
    {"nome": "Carregador",    "peso": 0.5, "valor": 150},
    {"nome": "Tablet",        "peso": 1.0, "valor": 1200},
]

PESO_MAXIMO = 7.0
NUM_ITENS = len(itens)

for item in itens:
    print(" ", item["nome"], "\t", item["peso"], "kg \tR$", item["valor"])
print("\nPeso maximo:", PESO_MAXIMO, "kg")

  Notebook 	 3.0 kg 	R$ 2000
  Livro de IA 	 1.5 kg 	R$ 300
  Garrafa agua 	 1.0 kg 	R$ 50
  Camera 	 2.0 kg 	R$ 1500
  Casaco 	 1.5 kg 	R$ 200
  Lanche 	 0.5 kg 	R$ 80
  Carregador 	 0.5 kg 	R$ 150
  Tablet 	 1.0 kg 	R$ 1200

Peso maximo: 7.0 kg


---
## Passo 2: Definir tipos com o `creator`

No DEAP, primeiro definimos **tipos personalizados**:
- `FitnessMax`: tipo de fitness que queremos **maximizar**
- `Individual`: tipo de individuo (lista com fitness acoplado)

O parametro `weights` define a direcao:
- `weights=(1.0,)` -> **maximizar** (nosso caso)
- `weights=(-1.0,)` -> minimizar
- `weights=(1.0, -1.0)` -> multi-objetivo (maximiza 1, minimiza outro)

In [14]:
# Criar tipo de fitness: maximizar 1 objetivo
creator.create("FitnessMax", base.Fitness, weights=(1.0,))

# Criar tipo de individuo: lista com fitness acoplado
creator.create("Individual", list, fitness=creator.FitnessMax)

print("Tipos criados!")
print("  FitnessMax: maximizar 1 objetivo")
print("  Individual: lista com fitness")

Tipos criados!
  FitnessMax: maximizar 1 objetivo
  Individual: lista com fitness


C:\Users\Aluno\AppData\Roaming\Python\Python314\site-packages\deap\creator.py:185: RuntimeWarning: A class named 'FitnessMax' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "
C:\Users\Aluno\AppData\Roaming\Python\Python314\site-packages\deap\creator.py:185: RuntimeWarning: A class named 'Individual' has already been created and it will be overwritten. Consider deleting previous creation of that class or rename it.
  warnings.warn("A class named '{0}' has already been created and it "


---
## Passo 3: Registrar operadores no `toolbox`

O toolbox e a **caixa de ferramentas** do DEAP. Registramos cada operador com um nome. Isso da **flexibilidade total** para trocar operadores facilmente.

In [15]:
toolbox = base.Toolbox()

# Gerador de genes: cada gene e 0 ou 1
toolbox.register("gene", random.randint, 0, 1)

# Gerador de individuo: lista de genes
toolbox.register("individual", tools.initRepeat, creator.Individual,
                 toolbox.gene, n=NUM_ITENS)

# Gerador de populacao: lista de individuos
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

# Mostrar um individuo de exemplo
exemplo = toolbox.individual()
print("Individuo de exemplo:", exemplo)
print("Tipo:", type(exemplo))

Individuo de exemplo: [0, 1, 0, 0, 0, 1, 0, 0]
Tipo: <class 'deap.creator.Individual'>


---
## Passo 4: Funcao de fitness

No DEAP, o fitness deve retornar uma **tupla** `(valor,)` — mesmo com 1 objetivo. Isso porque o DEAP suporta **multi-objetivo** nativamente.

In [16]:
def funcao_fitness(individuo):
    """Retorna uma TUPLA com o valor total (0 se excedeu peso)."""
    peso_total = 0
    valor_total = 0

    for i in range(NUM_ITENS):
        if individuo[i] == 1:
            peso_total += itens[i]["peso"]
            valor_total += itens[i]["valor"]

    if peso_total > PESO_MAXIMO:
        return (0,)          # TUPLA! Obrigatorio no DEAP

    return (valor_total,)    # TUPLA!

# Registrar no toolbox
toolbox.register("evaluate", funcao_fitness)

# Testar
teste = toolbox.individual()
print("Individuo:", teste)
print("Fitness:", funcao_fitness(teste))

Individuo: [1, 1, 1, 1, 0, 0, 0, 0]
Fitness: (0,)


---
## Passo 5: Registrar operadores geneticos

Aqui registramos selecao, crossover e mutacao. A **vantagem** do DEAP: trocar qualquer operador e so mudar UMA linha!

### Operadores disponiveis:

| Tipo | Opcoes disponiveis |
|:---|:---|
| **Selecao** | `selTournament`, `selRoulette`, `selBest`, `selRandom` |
| **Crossover** | `cxOnePoint`, `cxTwoPoint`, `cxUniform` |
| **Mutacao** | `mutFlipBit`, `mutGaussian`, `mutShuffleIndexes` |

In [40]:
# Selecao por torneio (sorteia 3, pega o melhor)
toolbox.register("select", tools.selTournament, tournsize=3)
#toolbox.register("select", tools.selRoulette)
#toolbox.register("select", tools.selBest)
#toolbox.register("select", tools.selRandom)

# Crossover de um ponto
toolbox.register("mate", tools.cxOnePoint)

# Mutacao: inverte cada bit com probabilidade 5%
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)

print("Operadores registrados:")
print("  Selecao:   selTournament (torneio de 3)")
print("  Crossover: cxOnePoint (1 ponto)")
print("  Mutacao:   mutFlipBit (5% por gene)")

Operadores registrados:
  Selecao:   selTournament (torneio de 3)
  Crossover: cxOnePoint (1 ponto)
  Mutacao:   mutFlipBit (5% por gene)


---
## Passo 6: Rodar o AG!

O DEAP tem o `eaSimple` que faz tudo automaticamente e mostra uma **tabela de evolucao** com estatisticas por geracao.

O **Hall of Fame** guarda os melhores individuos (equivalente ao elitismo).

In [41]:
# Criar populacao inicial
populacao = toolbox.population(n=20)

# Hall of Fame: guarda o melhor individuo
hall_of_fame = tools.HallOfFame(1)

# Estatisticas para acompanhar
stats = tools.Statistics(lambda ind: ind.fitness.values[0])
stats.register("media", lambda x: round(sum(x) / len(x)))
stats.register("melhor", max)

# Rodar!
populacao_final, log = algorithms.eaSimple(
    populacao,
    toolbox,
    cxpb=0.8,         # Taxa de crossover
    mutpb=0.2,         # Taxa de mutacao
    ngen=50,           # Geracoes
    stats=stats,
    halloffame=hall_of_fame,
    verbose=True
)

gen	nevals	media	melhor
0  	20    	2018 	3700  
1  	12    	2946 	4750  
2  	14    	3284 	4850  
3  	17    	3566 	4850  
4  	18    	3930 	4850  
5  	16    	4684 	4850  
6  	17    	4815 	4850  
7  	17    	4598 	4850  
8  	18    	4750 	4850  
9  	18    	4854 	4930  
10 	12    	4862 	4930  
11 	18    	4890 	4930  
12 	18    	4914 	4930  
13 	20    	4684 	4930  
14 	18    	4930 	4930  
15 	20    	4930 	4930  
16 	19    	4680 	4930  
17 	13    	4930 	4930  
18 	16    	4437 	4930  
19 	13    	4926 	4930  
20 	16    	4524 	4930  
21 	19    	4123 	4930  
22 	13    	4762 	4930  
23 	18    	4433 	4930  
24 	13    	4684 	4930  
25 	16    	4684 	4930  
26 	20    	4680 	4930  
27 	18    	4922 	4930  
28 	15    	4684 	4930  
29 	16    	4930 	4930  
30 	20    	4930 	4930  
31 	19    	4524 	4930  
32 	16    	4830 	4930  
33 	15    	4830 	4930  
34 	14    	4930 	4930  
35 	17    	4680 	4930  
36 	16    	4684 	4930  
37 	17    	4608 	4930  
38 	15    	4684 	4930  
39 	18    	4437 	4930  
40 	15    	4930 

---
## Passo 7: Resultados

In [42]:
# Melhor individuo do Hall of Fame
melhor = hall_of_fame[0]

print("MELHOR SOLUCAO (DEAP)")
print("=" * 50)

peso_total = 0
valor_total = 0

for i in range(NUM_ITENS):
    if melhor[i] == 1:
        print("  [X]", itens[i]["nome"], "-", itens[i]["peso"], "kg - R$", itens[i]["valor"])
        peso_total += itens[i]["peso"]
        valor_total += itens[i]["valor"]
    else:
        print("  [ ]", itens[i]["nome"])

print()
print("  Cromossomo:", list(melhor))
print("  Peso total:", peso_total, "kg")
print("  Valor total: R$", valor_total)

MELHOR SOLUCAO (DEAP)
  [X] Notebook - 3.0 kg - R$ 2000
  [ ] Livro de IA
  [ ] Garrafa agua
  [X] Camera - 2.0 kg - R$ 1500
  [ ] Casaco
  [X] Lanche - 0.5 kg - R$ 80
  [X] Carregador - 0.5 kg - R$ 150
  [X] Tablet - 1.0 kg - R$ 1200

  Cromossomo: [1, 0, 0, 1, 0, 1, 1, 1]
  Peso total: 7.0 kg
  Valor total: R$ 4930


---
## Comparacao final: 3 abordagens

| Aspecto | Na mao (Ex 02) | PyGAD (Ex 05) | DEAP (Ex 06) |
|:---|:---:|:---:|:---:|
| Linhas de codigo | ~120 | ~20 | ~50 |
| Funcoes criadas | 7 | 1 | 1 |
| Flexibilidade | Total | Media | **Alta** |
| Multi-objetivo | Manual | Nao | **Sim** |
| Artigos cientificos | - | Poucos | **Muitos** |
| Dificuldade | Alta | Baixa | Media |

### Recomendacao:

- **Aprender AG** -> Na mao (Exemplos 01-04)
- **Resolver rapido** -> PyGAD (Exemplo 05)
- **Pesquisa / artigo** -> DEAP (Exemplo 06)

### Exercicio:

Troque `tools.selTournament` por `tools.selRoulette` e compare os resultados!